# Gravity-Based Accessibility Model — Phase 2

This notebook computes gravity-based multimodal accessibility scores for a set of origin grid cells, combining walking, public transport, and e-bike legs. 

---

## Folder Structure

All input files must be placed in a `data/` folder located next to this notebook. 

```
Phase 2/
├── compute_gravity.ipynb   ← this notebook
├── create_pt_times_via_r5py.ipynb        ← run this first to generate pt_times.csv
├── requirements.txt
└── data/
    ├── lux_walk.graphml
    ├── lux_bike.graphml
    ├── stops_only_luxemburg.csv
    ├── bike_stations_merged.csv
    ├── origins.csv
    ├── pois.csv
    └── pt_times.csv                      ← generated by create_pt_times_via_r5py.ipynb
```

> **Note:** Run `create_pt_times_via_r5py.ipynb` first to generate `pt_times.csv` before running this notebook.

---

## Required Columns per Input File

| File | Required Columns |
|---|---|
| `origins.csv` | `GRD_ID`, `x`, `y` |
| `stops_only_luxemburg.csv` | `stop_id`, `lat`, `lon` |
| `bike_stations_merged.csv` | `longitude`, `latitude` |
| `pois.csv` | `x`, `y`, `category` |
| `pt_times.csv` | `stop_id_from`, `stop_id_to`, `pt_time_min` |
| `lux_walk.graphml` | created at Phase 1  
| `lux_bike.graphml` | created at Phase 1


---

## Installation

```bash
pip install -r requirements.txt
```

## Output Files

All outputs are written to folders next to this notebook (configurable in cell 0).

| File | Description |
|---|---|
| `accessibility_results_sigma_{sigma}.csv` | Contains raw accessibility scores (`A_<category>_<scenario>`) and delta columns (`dA_<category>_B/C/D`) for each POI category and scenario. |
| `delta_summary_sigma_{sigma}.csv` | Summary statistics (mean, median, % positive, min, max delta) per POI category and scenario. |




In [1]:
import os
import sys
import math
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Set, Tuple

import geopandas as gpd
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
from tqdm import tqdm

print("all good")

all good


In [5]:

# ============================================================
# 0. CONFIGURATION
# ============================================================

# --- Project root ---
PROJECT_ROOT = Path(__file__).parent if "__file__" in dir() else Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUT_DIR   = PROJECT_ROOT / "phase2_outputs_sigma_tests"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Graph files from Phase 1  ---
PHASE1_DIR   = PROJECT_ROOT.parent / "Phase 1" / "cache"
WALK_GRAPH   = str(PHASE1_DIR / "lux_walk.graphml")
BIKE_GRAPH   = str(PHASE1_DIR / "lux_bike.graphml")

# GTFS stops only in Luxembourg
GTFS_STOPS   = str(DATA_DIR / "stops_only_luxemburg.csv")
STATIONS_CSV = str(DATA_DIR / "bike_stations_merged.csv")
ORIGINS_CSV  = str(DATA_DIR / "origins.csv")
POIS_CSV     = str(DATA_DIR / "pois.csv")
PT_TIMES_CSV = str(DATA_DIR / "pt_times.csv")

CRS_PROJ = "EPSG:2169"
CRS_GEO = "EPSG:4326"

# Network walk-time thresholds (minutes)
WALK_STOP = 10.0
WALK_STATION = 8.0

# Caps on candidates retained after network search
MAX_CANDIDATE_STOPS = 12
MAX_CANDIDATE_STATIONS = 12

# Large penalty for infeasible paths
INF = 1e9

# One common sigma for all categories
SIGMA_VALUES = [15.0, 20.0, 25.0]

POI_CATEGORIES = [
    "Entertainment",
    "Jobs",
    "Commercial",
    "Bars and restaurants",
    "Food",
    "Education",
    "Parks",
    "Basic Healthcare",
]

SCENARIOS = ["A", "B", "C", "D"]


# ============================================================
# 1. DATA STRUCTURES
# ============================================================

@dataclass(frozen=True)
class Candidate:
    """A reachable stop or station with its network walk time."""
    entity_id: str
    time_min: float

print("All good!")


All good!


In [6]:
# ============================================================
# 2. UTILITY FUNCTIONS
# ============================================================

def gaussian_decay(t_min: float, sigma: float) -> float:
    if t_min >= INF:
        return 0.0
    return float(np.exp(-(t_min ** 2) / (2.0 * sigma ** 2)))


def required_columns(df: pd.DataFrame, cols: List[str], name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")


def stop_id_of_row(row: pd.Series, fallback_idx: int) -> str:
    if "stop_id" in row.index:
        return str(row["stop_id"])
    return str(fallback_idx)


def station_id_of_row(row: pd.Series, fallback_idx: int) -> str:
    for candidate in ["station_id", "id", "number", "station_no"]:
        if candidate in row.index:
            return str(row[candidate])
    return str(fallback_idx)


def single_source_walk(G: nx.MultiDiGraph, source_node, cutoff: float) -> Dict:
    try:
        return nx.single_source_dijkstra_path_length(
            G, source_node, cutoff=cutoff, weight="walk_time_min"
        )
    except nx.NodeNotFound:
        return {}


def safe_bike_path(G: nx.MultiDiGraph, source, target) -> float:
    try:
        return float(nx.shortest_path_length(G, source, target, weight="bike_time_min"))
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return INF


print("All good!")


All good!


In [7]:
# ============================================================
# 3.1  LOAD AND VALIDATE NETWORK DATA + ASSIGN SLOPE-ADJUSTED TRAVEL TIMES
# ============================================================

print("[1] Loading and validating data...")

for path in [WALK_GRAPH, BIKE_GRAPH, GTFS_STOPS, STATIONS_CSV, ORIGINS_CSV, POIS_CSV, PT_TIMES_CSV]:
    if not os.path.exists(path):
        sys.exit(f"\nRequired file not found: {path}\n") ### make it Fatma

G_walk = ox.load_graphml(WALK_GRAPH)
G_bike = ox.load_graphml(BIKE_GRAPH)

# ------------------------------------------------------------
# Reuse the slope-adjusted impedance specification from Phase 1
# ------------------------------------------------------------
STAIR_SPEED_M_PER_MIN = 42.0

BASE_KMH = 18.0
MIN_KMH = 6.0
UPHILL_THRESHOLD_PCT = 3.0
PENALTY_PER_PCT = 1.5
DOWNHILL_CAP_KMH = 24.0


def tobler_speed_kmh(grade_dec: float) -> float:
    """Tobler hiking function used in Phase 1."""
    return 6.0 * math.exp(-3.5 * abs(grade_dec + 0.05))


def ebike_speed_kmh(grade_dec: float) -> float:
    """Slope-adjusted e-bike speed rule used in Phase 1."""
    g_pct = grade_dec * 100.0
    if g_pct >= 0:
        excess = max(0.0, g_pct - UPHILL_THRESHOLD_PCT)
        speed = BASE_KMH - PENALTY_PER_PCT * excess
        return max(MIN_KMH, speed)
    else:
        benefit = 1.0 + 0.02 * min(10.0, abs(g_pct))
        return min(DOWNHILL_CAP_KMH, BASE_KMH * benefit)


def drop_bad_edges_inplace(G: nx.MultiDiGraph, time_attr: str) -> None:
    """Remove edges where the requested travel-time attribute is missing or invalid."""
    bad_edges = []
    for u, v, k, data in G.edges(keys=True, data=True):
        t = data.get(time_attr)
        if t is None:
            bad_edges.append((u, v, k))
            continue
        try:
            t = float(t)
        except (TypeError, ValueError):
            bad_edges.append((u, v, k))
            continue
        if not np.isfinite(t):
            bad_edges.append((u, v, k))

    if bad_edges:
        G.remove_edges_from(bad_edges)
    print(f"  Dropped {len(bad_edges)} edges with invalid '{time_attr}'")


print("Assigning slope-adjusted walk_time_min to walk graph edges...")
walk_missing_grade = 0
for u, v, k, data in G_walk.edges(keys=True, data=True):
    length_m = float(data.get("length", 0.0) or 0.0)
    grade = float(data.get("grade", 0.0) or 0.0)
    if "grade" not in data:
        walk_missing_grade += 1

    highway = data.get("highway")
    is_steps = ("steps" in highway) if isinstance(highway, list) else (highway == "steps")

    if length_m <= 0:
        data["walk_time_min"] = None
        continue

    if is_steps:
        data["walk_time_min"] = length_m / STAIR_SPEED_M_PER_MIN
    else:
        speed_kmh = tobler_speed_kmh(grade)
        data["walk_time_min"] = ((length_m / 1000.0) / speed_kmh) * 60.0

print("Assigning slope-adjusted bike_time_min to bike graph edges...")
bike_missing_grade = 0
for u, v, k, data in G_bike.edges(keys=True, data=True):
    length_m = float(data.get("length", 0.0) or 0.0)
    grade = float(data.get("grade", 0.0) or 0.0)
    if "grade" not in data:
        bike_missing_grade += 1

    if length_m <= 0:
        data["bike_time_min"] = None
        continue

    speed_kmh = ebike_speed_kmh(grade)
    data["bike_time_min"] = ((length_m / 1000.0) / speed_kmh) * 60.0

drop_bad_edges_inplace(G_walk, "walk_time_min")
drop_bad_edges_inplace(G_bike, "bike_time_min")

if walk_missing_grade > 0 or bike_missing_grade > 0:
    print("WARNING: Some edges are missing a 'grade' attribute and were treated as grade = 0.")
    print(f"  Walk edges without grade: {walk_missing_grade}")
    print(f"  Bike edges without grade: {bike_missing_grade}")

sample_walk = list(G_walk.edges(data=True))[0]
sample_bike = list(G_bike.edges(data=True))[0]
print("Walk edge walk_time_min:", sample_walk[2]["walk_time_min"])
print("Bike edge bike_time_min:", sample_bike[2]["bike_time_min"])
print("  Done.")

[1] Loading and validating data...
Assigning slope-adjusted walk_time_min to walk graph edges...
Assigning slope-adjusted bike_time_min to bike graph edges...
  Dropped 0 edges with invalid 'walk_time_min'
  Dropped 0 edges with invalid 'bike_time_min'
Walk edge walk_time_min: 0.15980065575203722
Bike edge bike_time_min: 0.044715260798190284
  Done.


In [8]:
# ============================================================
# 3.2  LOAD AND VALIDATE DATA 
# ============================================================

print("[1] Loading and validating data...")

origins = pd.read_csv(ORIGINS_CSV)
stops_raw = pd.read_csv(GTFS_STOPS, dtype={"stop_id": str})
stations_raw = pd.read_csv(STATIONS_CSV)
pois = pd.read_csv(POIS_CSV)
pt_times = pd.read_csv(PT_TIMES_CSV, dtype={"stop_id_from": str, "stop_id_to": str})

required_columns(origins, ["GRD_ID", "x", "y"], "origins")
required_columns(stops_raw, ["stop_id", "lat", "lon"], "gtfs stops")
required_columns(pois, ["x", "y", "category"], "pois")
required_columns(stations_raw, ["longitude", "latitude"], "stations")
required_columns(pt_times, ["stop_id_from", "stop_id_to", "pt_time_min"], "pt_times")

# GTFS stops are geographic; project them to match network coordinates
stops = gpd.GeoDataFrame(
    stops_raw.copy(),
    geometry=gpd.points_from_xy(stops_raw["lon"], stops_raw["lat"]),
    crs=CRS_GEO,
).to_crs(CRS_PROJ)

stops["x"] = stops.geometry.x
stops["y"] = stops.geometry.y

stations = gpd.GeoDataFrame(
    stations_raw,
    geometry=gpd.points_from_xy(stations_raw["longitude"], stations_raw["latitude"]),
    crs=CRS_GEO,
).to_crs(CRS_PROJ)

stations["x"] = stations.geometry.x
stations["y"] = stations.geometry.y

origins = origins.copy().reset_index(drop=True)
stops = stops.copy().reset_index(drop=True)
pois = pois.copy().reset_index(drop=True)
stations = stations.copy().reset_index(drop=True)
pt_times = pt_times.copy().reset_index(drop=True)

origins["origin_id"] = origins["GRD_ID"].astype(str)
stops["stop_id_str"] = [stop_id_of_row(row, i) for i, row in stops.iterrows()]
stations["station_id_str"] = [station_id_of_row(row, i) for i, row in stations.iterrows()]
pois["poi_id"] = pois.index.astype(int)

if stops["stop_id_str"].duplicated().any():
    dupes = stops.loc[stops["stop_id_str"].duplicated(), "stop_id_str"].tolist()[:10]
    raise ValueError(f"Duplicate stop_id_str values found, e.g. {dupes}")

if stations["station_id_str"].duplicated().any():
    dupes = stations.loc[stations["station_id_str"].duplicated(), "station_id_str"].tolist()[:10]
    raise ValueError(f"Duplicate station_id_str values found, e.g. {dupes}")

# Clean PT matrix
pt_times["pt_time_min"] = pd.to_numeric(pt_times["pt_time_min"], errors="coerce")
pt_times = pt_times.dropna(subset=["pt_time_min"]).copy()
pt_times = pt_times[pt_times["pt_time_min"] >= 0].copy()
pt_times["stop_id_from"] = pt_times["stop_id_from"].astype(str)
pt_times["stop_id_to"] = pt_times["stop_id_to"].astype(str)
pt_times = pt_times[pt_times["stop_id_from"] != pt_times["stop_id_to"]].copy()

# ── NEW: Filter PT matrix to Luxembourg-only stops ──────────
lux_stop_ids = set(stops["stop_id_str"])
pt_before = len(pt_times)
pt_times = pt_times[
    pt_times["stop_id_from"].isin(lux_stop_ids) &
    pt_times["stop_id_to"].isin(lux_stop_ids)
].copy().reset_index(drop=True)
print(f"  PT pairs filtered to Lux-only stops: {pt_before} → {len(pt_times)}")
# ─────────────────────────────────────────────────────────────

# Deduplicate stop-pairs defensively
pt_times = (
    pt_times.groupby(["stop_id_from", "stop_id_to"], as_index=False)["pt_time_min"]
    .median()
)

# Keep only GTFS stops that actually exist in the PT matrix
valid_pt_stops = set(pt_times["stop_id_from"]) | set(pt_times["stop_id_to"])
stops = stops[stops["stop_id_str"].isin(valid_pt_stops)].copy().reset_index(drop=True)

missing_categories = sorted(set(POI_CATEGORIES) - set(pois["category"].unique()))
extra_categories = sorted(set(pois["category"].unique()) - set(POI_CATEGORIES))
if missing_categories:
    print(f"WARNING: These configured POI categories are missing from pois.csv: {missing_categories}")
if extra_categories:
    print(f"WARNING: These categories exist in pois.csv but are not in POI_CATEGORIES: {extra_categories}")

missing_from_matrix = set(stops["stop_id_str"]) - (
    set(pt_times["stop_id_from"]) | set(pt_times["stop_id_to"])
)
print(f"  GTFS stops not present in PT matrix: {len(missing_from_matrix)}")

print(f"  Walk graph : {len(G_walk.nodes)} nodes, {len(G_walk.edges)} edges")
print(f"  Bike graph : {len(G_bike.nodes)} nodes, {len(G_bike.edges)} edges")
print(f"  Origins    : {len(origins)}")
print(f"  GTFS stops : {len(stops)}")
print(f"  Stations   : {len(stations)}")
print(f"  POIs       : {len(pois)}")
print(f"  PT pairs   : {len(pt_times)}")

[1] Loading and validating data...
  PT pairs filtered to Lux-only stops: 4964083 → 4764359
  GTFS stops not present in PT matrix: 0
  Walk graph : 99289 nodes, 254087 edges
  Bike graph : 99289 nodes, 246317 edges
  Origins    : 2795
  GTFS stops : 2522
  Stations   : 290
  POIs       : 13497
  PT pairs   : 4764359


In [9]:
# ============================================================
# 4. MAP ENTITIES TO GRAPH NODES
# ============================================================
# FIX: Pass entire coordinate arrays to ox.nearest_nodes at once.
# The original code called ox.nearest_nodes row-by-row, which
# rebuilt the internal KD-tree on every call (~98k nodes each time).
# Vectorised version: seconds instead of 35+ minutes.
# ============================================================

print("\n[2] Mapping entities to graph nodes...")

def map_to_walk_node(df: pd.DataFrame) -> pd.Series:
    nodes = ox.nearest_nodes(G_walk, df["x"].values, df["y"].values)
    return pd.Series(nodes, index=df.index)


def map_to_bike_node(df: pd.DataFrame) -> pd.Series:
    nodes = ox.nearest_nodes(G_bike, df["x"].values, df["y"].values)
    return pd.Series(nodes, index=df.index)


origins["walk_node"] = map_to_walk_node(origins)
stops["walk_node"] = map_to_walk_node(stops)
pois["walk_node"] = map_to_walk_node(pois)
stations["walk_node"] = map_to_walk_node(stations)
stations["bike_node"] = map_to_bike_node(stations)

walk_node_to_stop_ids: Dict[object, List[str]] = defaultdict(list)
for row in stops.itertuples():
    walk_node_to_stop_ids[row.walk_node].append(row.stop_id_str)

walk_node_to_station_ids: Dict[object, List[str]] = defaultdict(list)
for row in stations.itertuples():
    walk_node_to_station_ids[row.walk_node].append(row.station_id_str)

station_by_id = stations.set_index("station_id_str")

print("  Done.")





[2] Mapping entities to graph nodes...
  Done.


In [10]:
# ============================================================
# 5. PT LOOKUP
# ============================================================
# FIX: Use vectorised zip instead of iterrows() over ~5M rows.
# ============================================================

print("\n[3] Building PT travel time lookup...")

pt_lookup: Dict[Tuple[str, str], float] = dict(zip(
    zip(pt_times["stop_id_from"], pt_times["stop_id_to"]),
    pt_times["pt_time_min"].astype(float),
))
print(f"  PT stop-pair records: {len(pt_lookup)}")


def get_pt_time(stop_from: str, stop_to: str) -> float:
    return pt_lookup.get((stop_from, stop_to), INF)

print("  Done.")



[3] Building PT travel time lookup...
  PT stop-pair records: 4764359
  Done.


In [18]:
# ============================================================
# 6. CANDIDATE SET GENERATION
# ============================================================
# FIX: Combined stop+station candidate search into a single
# Dijkstra call per entity (was 2 calls each before).
# Uses max(TAU_WALK_STOP, TAU_WALK_STATION) as the single cutoff,
# then filters by per-entity threshold.
# ============================================================

print("\n[4] Building candidate stop and station sets...")

COMBINED_CUTOFF = max(WALK_STOP, WALK_STATION)


def get_candidates_combined(source_walk_node) -> Tuple[List[Candidate], List[Candidate]]:
    """Single Dijkstra, then split results into stop and station candidates."""
    reachable = single_source_walk(G_walk, source_walk_node, COMBINED_CUTOFF)
    stop_results: List[Candidate] = []
    station_results: List[Candidate] = []
    for node, t in reachable.items():
        t_float = float(t)
        if t_float <= WALK_STOP:
            for sid in walk_node_to_stop_ids.get(node, []):
                stop_results.append(Candidate(entity_id=sid, time_min=t_float))
        if t_float <= WALK_STATION:
            for sid in walk_node_to_station_ids.get(node, []):
                station_results.append(Candidate(entity_id=sid, time_min=t_float))
    stop_results.sort(key=lambda c: c.time_min)
    station_results.sort(key=lambda c: c.time_min)
    return stop_results[:MAX_CANDIDATE_STOPS], station_results[:MAX_CANDIDATE_STATIONS]


print("  Computing candidate stops and stations per origin...")
cand_stops_origin: Dict[str, List[Candidate]] = {}
cand_stations_Bo: Dict[str, List[Candidate]] = {}
for row in tqdm(origins.itertuples(), total=len(origins)):
    oid = row.origin_id
    s, st = get_candidates_combined(row.walk_node)
    cand_stops_origin[oid] = s
    cand_stations_Bo[oid] = st

print("  Computing candidate stops and stations per POI...")
cand_stops_poi: Dict[int, List[Candidate]] = {}
cand_stations_Bd: Dict[int, List[Candidate]] = {}
for row in tqdm(pois.itertuples(), total=len(pois)):
    pid = int(row.poi_id)
    s, st = get_candidates_combined(row.walk_node)
    cand_stops_poi[pid] = s
    cand_stations_Bd[pid] = st

print("  Computing candidate stations per PT stop (B(p+) = B(p-))...")
cand_stations_Bp: Dict[str, List[Candidate]] = {}
for row in tqdm(stops.itertuples(), total=len(stops)):
    sid = row.stop_id_str
    # Stops only need station candidates, but we reuse the combined
    # function (the stop results are simply discarded)
    _, st = get_candidates_combined(row.walk_node)
    cand_stations_Bp[sid] = st


print("  Done.")


[4] Building candidate stop and station sets...
  Computing candidate stops and stations per origin...


100%|██████████| 2795/2795 [00:01<00:00, 1560.06it/s]


  Computing candidate stops and stations per POI...


100%|██████████| 13497/13497 [00:20<00:00, 649.57it/s] 


  Computing candidate stations per PT stop (B(p+) = B(p-))...


100%|██████████| 2522/2522 [00:02<00:00, 901.01it/s] 

  Done.


In [19]:
# ============================================================
# 7. PRECOMPUTE BIKE TIMES BETWEEN REQUIRED STATION PAIRS
# ============================================================
# FIX: Use single-source Dijkstra per unique source station
# instead of one shortest_path_length call per pair.
# ============================================================

print("\n[5] Identifying required station pairs...")

required_pairs: Set[Tuple[str, str]] = set()

for oid, bo_list in cand_stations_Bo.items():
    for board_stop in cand_stops_origin[oid]:
        bp_list = cand_stations_Bp.get(board_stop.entity_id, [])
        for s_o in bo_list:
            for s_p in bp_list:
                required_pairs.add((s_o.entity_id, s_p.entity_id))

for pid, bd_list in cand_stations_Bd.items():
    for alight_stop in cand_stops_poi[pid]:
        bm_list = cand_stations_Bp.get(alight_stop.entity_id, [])
        for s_m in bm_list:
            for s_d in bd_list:
                required_pairs.add((s_m.entity_id, s_d.entity_id))

print(f"  Required station pairs: {len(required_pairs)}")

# Group pairs by source station for batched Dijkstra
source_to_targets: Dict[str, Set[str]] = defaultdict(set)
for s_from, s_to in required_pairs:
    source_to_targets[s_from].add(s_to)

print(f"  Unique source stations: {len(source_to_targets)}")
print("  Computing bike times via single-source Dijkstra per source station...")

bike_pair_times: Dict[Tuple[str, str], float] = {}

for s_from in tqdm(sorted(source_to_targets.keys())):
    src_node = station_by_id.loc[s_from, "bike_node"]
    targets = source_to_targets[s_from]

    # Compute shortest path from this source to ALL reachable bike nodes
    try:
        lengths = nx.single_source_dijkstra_path_length(
            G_bike, src_node, weight="bike_time_min"
        )
    except nx.NodeNotFound:
        for s_to in targets:
            bike_pair_times[(s_from, s_to)] = INF
        continue

    for s_to in targets:
        tgt_node = station_by_id.loc[s_to, "bike_node"]
        bike_pair_times[(s_from, s_to)] = float(lengths.get(tgt_node, INF))

print(f"  Bike pair times computed: {len(bike_pair_times)}")





[5] Identifying required station pairs...
  Required station pairs: 3593
  Unique source stations: 282
  Computing bike times via single-source Dijkstra per source station...


100%|██████████| 282/282 [02:10<00:00,  2.16it/s]

  Bike pair times computed: 3593


In [20]:
# ============================================================
# 8. PRECOMPUTE FOUR COST TABLES
# ============================================================

print("\n[6] Precomputing access and egress cost tables...")

access_walk: Dict[Tuple[str, str], float] = {}
access_bike: Dict[Tuple[str, str], float] = {}
egress_walk: Dict[Tuple[str, int], float] = {}
egress_bike: Dict[Tuple[str, int], float] = {}

print("  access_walk and access_bike...")
for orig_row in tqdm(origins.itertuples(), total=len(origins)):
    oid = orig_row.origin_id
    bo_list = cand_stations_Bo[oid]

    for stop_c in cand_stops_origin[oid]:
        k = stop_c.entity_id
        access_walk[(oid, k)] = stop_c.time_min

        if bo_list:
            bp_list = cand_stations_Bp.get(k, [])
            best = INF
            for s_o in bo_list:
                for s_p in bp_list:
                    t_bike = bike_pair_times.get((s_o.entity_id, s_p.entity_id), INF)
                    if t_bike >= INF:
                        continue
                    cost = s_o.time_min + t_bike + s_p.time_min
                    if cost < best:
                        best = cost
            access_bike[(oid, k)] = best
        else:
            access_bike[(oid, k)] = INF

print("  egress_walk and egress_bike...")
for poi_row in tqdm(pois.itertuples(), total=len(pois)):
    pid = int(poi_row.poi_id)
    bd_list = cand_stations_Bd[pid]

    for stop_c in cand_stops_poi[pid]:
        lid = stop_c.entity_id
        egress_walk[(lid, pid)] = stop_c.time_min

        if bd_list:
            bm_list = cand_stations_Bp.get(lid, [])
            best = INF
            for s_m in bm_list:
                for s_d in bd_list:
                    t_bike = bike_pair_times.get((s_m.entity_id, s_d.entity_id), INF)
                    if t_bike >= INF:
                        continue
                    cost = s_m.time_min + t_bike + s_d.time_min
                    if cost < best:
                        best = cost
            egress_bike[(lid, pid)] = best
        else:
            egress_bike[(lid, pid)] = INF

print("  Cost tables complete.")





[6] Precomputing access and egress cost tables...
  access_walk and access_bike...


100%|██████████| 2795/2795 [00:00<00:00, 81421.02it/s]


  egress_walk and egress_bike...


100%|██████████| 13497/13497 [00:00<00:00, 15746.85it/s]

  Cost tables complete.


In [21]:
# ============================================================
# 9-11. MAIN LOOP: ACCESSIBILITY + SAVE 
# ============================================================
# FIX: Merged cells 9, 10, 11 into one cell so that saving
# happens inside the sigma loop (previously only the last sigma
# was saved).
# FIX: Replaced iterrows() with itertuples() in inner loops.
# FIX: Pre-extract POI data as lists-of-tuples for faster
# iteration (avoids DataFrame overhead in the hot loop).
# ============================================================

print("\n[7] Computing gravity-based accessibility for all sigma values...")

categories = [cat for cat in POI_CATEGORIES if cat in set(pois["category"].unique())]
poi_by_category = {cat: pois[pois["category"] == cat] for cat in categories}

# Pre-extract POI ids per category as a plain list for fast iteration
poi_ids_by_category: Dict[str, List[int]] = {
    cat: df["poi_id"].astype(int).tolist()
    for cat, df in poi_by_category.items()
}

for sigma_common in SIGMA_VALUES:
    sigma_label = str(sigma_common).replace(".", "p")
    print(f"\n===== Running full model with common sigma = {sigma_common} =====")

    results = {
        cat: {sc: np.zeros(len(origins), dtype=float) for sc in SCENARIOS}
        for cat in categories
    }

    for cat in categories:
        pid_list = poi_ids_by_category[cat]
        print(f"\n  Category: {cat} | {len(pid_list)} POIs | sigma={sigma_common} min")

        if not pid_list:
            print(f"  WARNING: No POIs for category '{cat}'. Skipping.")
            continue

        for i, orig_row in enumerate(tqdm(
            origins.itertuples(),
            total=len(origins),
            desc=f"    {cat} | sigma={sigma_common}"
        )):
            oid = orig_row.origin_id
            board_stops = cand_stops_origin.get(oid, [])
            if not board_stops:
                continue

            for pid in pid_list:
                alight_stops = cand_stops_poi.get(pid, [])
                if not alight_stops:
                    continue

                best_A = INF
                best_B = INF
                best_C = INF
                best_D = INF

                for board in board_stops:
                    k = board.entity_id
                    t_aw = access_walk.get((oid, k), INF)
                    t_ab = access_bike.get((oid, k), INF)

                    for alight in alight_stops:
                        lid = alight.entity_id
                        t_pt = get_pt_time(k, lid)
                        if t_pt >= INF:
                            continue

                        t_ew = egress_walk.get((lid, pid), INF)
                        t_eb = egress_bike.get((lid, pid), INF)

                        if t_aw < INF and t_ew < INF:
                            tA = t_aw + t_pt + t_ew
                            if tA < best_A:
                                best_A = tA

                        if t_ab < INF and t_ew < INF:
                            tB = t_ab + t_pt + t_ew
                            if tB < best_B:
                                best_B = tB

                        if t_aw < INF and t_eb < INF:
                            tC = t_aw + t_pt + t_eb
                            if tC < best_C:
                                best_C = tC

                        if t_ab < INF and t_eb < INF:
                            tD = t_ab + t_pt + t_eb
                            if tD < best_D:
                                best_D = tD

                # A, B, C, D as parallel alternatives and focus on the deltas against A
                best_B = min(best_A, best_B)
                best_C = min(best_A, best_C)
                best_D = min(best_A, best_B, best_C, best_D)

                results[cat]["A"][i] += gaussian_decay(best_A, sigma_common)
                results[cat]["B"][i] += gaussian_decay(best_B, sigma_common)
                results[cat]["C"][i] += gaussian_decay(best_C, sigma_common)
                results[cat]["D"][i] += gaussian_decay(best_D, sigma_common)

    print("\n  Accessibility computation complete for this sigma.")

    # ========================================================
    # 10. SAVE OUTPUTS FOR THIS SIGMA
    # ========================================================

    print(f"\n[8] Saving outputs for sigma = {sigma_common}...")

    output = origins[["GRD_ID", "x", "y"]].copy()

    for cat in categories:
        slug = cat.lower().replace(" ", "_")
        for sc in SCENARIOS:
            output[f"A_{slug}_{sc}"] = results[cat][sc]
        output[f"dA_{slug}_B"] = results[cat]["B"] - results[cat]["A"]
        output[f"dA_{slug}_C"] = results[cat]["C"] - results[cat]["A"]
        output[f"dA_{slug}_D"] = results[cat]["D"] - results[cat]["A"]

    access_path = str(OUTPUT_DIR20 / f"accessibility_results_sigma_{sigma_label}.csv")
    output.to_csv(access_path, index=False)
    print(f"  Saved: {access_path}")

    summary_rows = []
    for cat in categories:
        slug = cat.lower().replace(" ", "_")
        for sc in ["B", "C", "D"]:
            col = f"dA_{slug}_{sc}"
            vals = output[col]
            summary_rows.append(
                {
                    "sigma": sigma_common,
                    "category": cat,
                    "scenario": sc,
                    "mean_delta": float(vals.mean()),
                    "median_delta": float(vals.median()),
                    "pct_positive": float((vals > 0).mean() * 100.0),
                    "max_delta": float(vals.max()),
                    "min_delta": float(vals.min()),
                }
            )

    summary = pd.DataFrame(summary_rows)
    summary_path = str(OUTPUT_DIR20 / f"delta_summary_sigma_{sigma_label}.csv")
    summary.to_csv(summary_path, index=False)
    print(f"  Saved: {summary_path}")
    print(summary.to_string(index=False))

  


[7] Computing gravity-based accessibility for all sigma values...

===== Running full model with common sigma = 15.0 =====

  Category: Entertainment | 3850 POIs | sigma=15.0 min


    Entertainment | sigma=15.0: 100%|██████████| 2795/2795 [02:56<00:00, 15.85it/s]



  Category: Jobs | 2918 POIs | sigma=15.0 min


    Jobs | sigma=15.0: 100%|██████████| 2795/2795 [02:35<00:00, 18.02it/s]



  Category: Commercial | 2623 POIs | sigma=15.0 min


...


  Category: Parks | 381 POIs | sigma=25.0 min


    Parks | sigma=25.0: 100%|██████████| 2795/2795 [00:14<00:00, 189.80it/s]



  Category: Basic Healthcare | 313 POIs | sigma=25.0 min


    Basic Healthcare | sigma=25.0: 100%|██████████| 2795/2795 [00:28<00:00, 99.03it/s] 



  Accessibility computation complete for this sigma.

[8] Saving outputs for sigma = 25.0...
  Saved: /Users/fgsumer/Desktop/thesis_code/Phase 2/phase2_outputs_sigma_tests_cap20/accessibility_results_sigma_25p0.csv
  Saved: /Users/fgsumer/Desktop/thesis_code/Phase 2/phase2_outputs_sigma_tests_cap20/delta_summary_sigma_25p0.csv
 sigma             category scenario  mean_delta  median_delta  pct_positive  max_delta  min_delta
  25.0        Entertainment        B    0.159436      0.000000      1.860465  56.201088        0.0
  25.0        Entertainment        C    0.375976      0.000369     57.173524   5.708211        0.0
  25.0        Entertainment        D    0.535905      0.000369     57.173524  61.077782        0.0
  25.0                 Jobs        B    0.159095      0.000000      1.788909  58.233985        0.0
  25.0                 Jobs        C    0.762703      0.001439     57.173524  13.408600        0.0
  25.0                 Jobs        D    0.921778      0.001439     57.173524

# Sensitivity for candidate cap 20 

In [17]:
OUTPUT_DIR20 = PROJECT_ROOT / "phase2_outputs_sigma_tests_cap20"
OUTPUT_DIR20.mkdir(parents=True, exist_ok=True)
MAX_CANDIDATE_STOPS = 20
MAX_CANDIDATE_STATIONS = 20

print(MAX_CANDIDATE_STOPS)
print(MAX_CANDIDATE_STATIONS)
# Run again from cell 6 onward


20
20
